# Semaine 9 — Jour 5 : Reranking, diversité et budget de contexte

Reclasser un ensemble limité de candidats selon la paire requête/passage.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. Les modèles et services optionnels ajoutent un temps d’installation distinct.

Prérequis : Python 3.11+, JSON, notions de vecteurs et de persistance. Le parcours obligatoire utilise uniquement la bibliothèque standard et ne contacte aucun service externe.

Lire [les objectifs](learning_objectives.md), [le chapitre](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week09/S9_J5_reranking.ipynb` et `notebooks/week09/teacher/S9_J5_reranking_teacher.ipynb`.

Critère de réussite : Assembler un contexte borné sans perdre les références des passages.

Les calculs de retrieval et de persistance sont exécutables; le corpus est fictif. L’encodeur de base est lexical et la réponse finale est extractive. Les extensions neuronales sont séparées et n’ont pas été exécutées ici.


# Objectifs mesurables

- Reclasser un ensemble limité de candidats selon la paire requête/passage.
- Distinguer score de retrieval, score de reranking et confiance de réponse.
- Assembler un contexte borné sans perdre les références des passages.

## Preuves attendues

Une sortie reproductible, un contrôle d’invariant et une limite du mécanisme correctement expliquée.


# S9 J5 — Reranking, diversité et budget de contexte


## 1. Retrouver puis reclasser
Le retriever sélectionne rapidement des candidats; le reranker les examine avec la requête. Cette séparation permet un traitement plus coûteux sur un petit ensemble. Un reranker ne recherche pas dans tout le corpus et ne peut pas découvrir un passage absent. Une amélioration du premier rang ne compense pas un rappel initial trop faible.

Le module obligatoire utilise une heuristique réelle et transparente : couverture des termes de la requête dans le texte, avec un petit bonus de couverture dans le titre. Elle calcule une fonction de la paire, mais n’est pas un CrossEncoder neuronal. L’adaptateur optionnel fournit une interface pour un modèle appris local non exécuté ici.

## 2. Interpréter le score
term_coverage est la proportion de termes de requête présents dans le passage. Le score ajoute 0,1 fois la couverture du titre. Un score élevé peut simplement révéler des mots communs; il ne démontre pas que le passage répond correctement. Le score du retriever est conservé sous retrieval_score afin de ne pas confondre les deux étapes.

Un CrossEncoder appris traite conjointement la requête et le candidat. Sa sortie peut être un logit, un score de pertinence ou une autre valeur selon le modèle. Elle ne devient pas automatiquement une probabilité de vérité. La calibration et l’évaluation restent nécessaires, y compris quand le modèle est préentraîné pour le reranking.

## 3. Sélectionner sous contrainte
context_pack ajoute les passages entiers tant que le budget de mots le permet. Il peut sauter un grand passage et conserver un passage plus court suivant. Le budget est en mots séparés par espaces, pas en tokens LLM. Il n’inclut pas le coût complet d’un prompt de génération; une intégration réelle doit compter instructions, question, wrappers et réserve de sortie.

Ne pas tronquer silencieusement un passage cité en conservant ses anciens offsets. Le mode local garde les extraits entiers et vérifiables. Si le budget ne contient aucun candidat suffisamment étayé, la réponse s’abstient. Une absence de preuve vaut mieux qu’une citation fabriquée.

## 4. Redondance et diversité
Plusieurs chunks d’un même document peuvent être pertinents mais redondants. Un chevauchement important amplifie ce phénomène. Le moteur de base n’implémente ni MMR ni déduplication sémantique; le challenge propose un packing limité à un passage par document pour étudier le compromis de diversité.

Une réponse peut aussi exiger deux passages complémentaires du même document. Imposer une seule source par document partout ferait perdre cette complémentarité. Mesurer le besoin avant d’appliquer une règle de diversité uniforme. Le projet conserve tous les identifiants de citations pour permettre cette inspection.

## 5. Un seuil d’abstention explicite
Le mode extractif exige une couverture minimale de 0,6, retire les chunks de titre isolés et applique le budget. Ce seuil est pédagogique et lexical. Il peut refuser une excellente paraphrase sans mots communs ou accepter un texte trompeur qui répète la requête. Il ne doit pas être présenté comme un détecteur fiable de factualité.

Pour un générateur réel, conserver les sources et vérifier séparément fidélité, citations et utilité. Un identifiant de citation valide prouve qu’un passage a été retrouvé; il ne prouve pas que toutes les affirmations ajoutées par un modèle sont soutenues. La réponse par défaut ne synthétise pas : elle reproduit les passages sélectionnés.

## 6. Expérience du jour
Observer l’ordre des candidats avant/après l’heuristique, mesurer les rangs documentaires et vérifier le contenu du contexte. Construire un exemple où le candidat manquant ne peut pas remonter. Montrer aussi qu’un budget trop faible force une abstention, sans transformer le manque de place en une prétendue absence de connaissance du corpus.

## À retenir
Le reranking affine la pertinence des candidats disponibles. Le packing et la fidélité des références font partie du contrat de réponse, sans devenir une preuve automatique de justesse.


## Diagramme du jour

```mermaid
flowchart TD
 A["Candidats du retrieval"] --> B["Score de paire"]
 B --> C["Ordre reclassé"]
 C --> D["Seuil lexical explicite"]
 D --> E["Budget et passages entiers"]
 E --> F["Contexte avec références"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

Les cellules s’exécutent hors ligne depuis l’arbre du dépôt.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week09/knowledge.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week09.knowledge import context_pack, rerank
candidates = [
    {"chunk_id": "long", "doc_id": "D1", "text": "un deux trois quatre cinq six sept huit neuf", "title": "", "score": .9},
    {"chunk_id": "medium", "doc_id": "D2", "text": "un deux trois quatre cinq", "title": "", "score": .8},
    {"chunk_id": "short", "doc_id": "D2", "text": "un deux trois", "title": "", "score": .7},
]
packed = context_pack(candidates, word_budget=8)
assert [c["chunk_id"] for c in packed] == ["medium", "short"]
ranked = rerank("quatre cinq", candidates)
assert ranked[0]["term_coverage"] == 1
print([c["chunk_id"] for c in ranked])

# Lab J5 — Reranking, diversité et budget de contexte

Depuis la racine du dépôt :

```bash
python3 book/week09/day05/labs/run.py
python3 -m examples.week09.demo --day 5
```

Résultat attendu : **classements avant/après heuristique; aucun gain universel supposé**.

## Manipulations

1. Reproduire la sortie et retrouver le calcul dans `examples/week09/knowledge.py`.
2. Réaliser l’exercice 2 avec une copie des données, sans modifier le corpus distribué.
3. Vérifier l’invariant étudié et expliquer une limite.

Le corpus se trouve dans `examples/week09/corpus`. Les tests automatisés sont sous `tests/week09`. [QUICKSTART](../../../../docs/week09/QUICKSTART.md) décrit les commandes et les codes retour.

Les modèles locaux et pgvector sont optionnels : lire `examples/week09/optional/README.md` depuis la racine du dépôt. Ils ne sont pas nécessaires pour finir cette journée et ne sont pas présentés comme validés dans cet environnement.


# Exercices

Conserver votre première réponse avant de lire les solutions.

## Exercice 1 — Relier les scores

Quelle différence entre retrieval_score et term_coverage ?

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 2 — Calculer le packing

Avec budget=8 mots et des passages de 9, 5 et 3 mots, quels passages sont retenus ?

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 3 — Étudier la diversité

Pourquoi un passage unique par document peut-il améliorer ou dégrader une réponse ?

Livrable : réponse argumentée avec calcul, test ou observation.


# Entretien

Répondre en deux minutes avec un exemple concret et une limite.

## Question 1

Un reranker peut-il réparer un défaut de rappel ?

## Question 2

Quand faut-il recalibrer un seuil de score ?


# Challenge

Écrire une variante de packing qui garde au plus un passage par document, avec le même budget en mots. Tester le gain de diversité et décrire le risque de perte de complémentarité.

## Critères de réussite

Reproductibilité, provenance des données, invariant vérifié et limite expliquée sans exagérer les garanties.


# Références — J5

Sources primaires consultées le 20 octobre 2026. Les algorithmes et contrats du lab sont explicités dans le chapitre; le corpus est une création fictive du cours.

- [pgvector : documentation officielle](https://github.com/pgvector/pgvector) — stockage, distances, recherche exacte/ANN et filtres.
- [Sentence Transformers : retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) — architecture de récupération en deux étapes.
- [SentenceTransformer : API](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html) — encode_query, encode_document et modèles locaux.
- [CrossEncoder : API](https://www.sbert.net/docs/package_reference/cross_encoder/model.html) — scoring conjoint de paires.
- [Cormack, Clarke et Buettcher, SIGIR 2009 : Reciprocal Rank Fusion](https://research.google/pubs/reciprocal-rank-fusion-outperforms-condorcet-and-individual-rank-learning-methods/) — origine de la méthode de fusion des rangs.
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html) — persistance locale et transactions.

Lire en priorité les références du thème du jour. Les extensions évoluent; leur environnement réel doit être verrouillé et testé avant utilisation. Le paquet n’affirme pas avoir exécuté de modèle neuronal ou de serveur pgvector.
